# 01 - Thu thập dữ liệu đánh giá Laptop trên FPTShop
Notebook này crawl dữ liệu đánh giá từ FPTShop theo hướng HTTP thuần (requests), chuẩn hóa schema tương thích pipeline EDA và lưu CSV UTF-8.

In [8]:
%pip install requests pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import json
import re
import time
from html import unescape
from urllib.parse import urljoin, urlparse, parse_qsl, urlencode
import hashlib

import pandas as pd
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/json;q=0.9,*/*;q=0.8",
    "Referer": "https://fptshop.com.vn/"
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)

COMMENT_API_URL = "https://papi.fptshop.com.vn/gw/v1/public/bff-before-order/comment/list"
COMMENT_API_HEADERS = {
    "Accept": "application/json",
    "Content-Type": "application/json",
    "Order-Channel": "1",
    "Referer": "https://fptshop.com.vn/",
}


def with_query_param(url: str, key: str, value):
    parsed = urlparse(url)
    query = dict(parse_qsl(parsed.query, keep_blank_values=True))
    query[key] = str(value)
    return parsed._replace(query=urlencode(query)).geturl()


def collect_product_urls(
    search_url: str,
    max_pages_safety: int = 300,
    max_products: int | None = None,
    hard_cap: int | None = None,
    consecutive_empty_pages: int = 3,
    delay_seconds: float = 0.8,
):
    product_urls = []
    seen = set()
    empty_pages = 0

    for page in range(1, max_pages_safety + 1):
        page_url = with_query_param(search_url, "page", page)
        response = SESSION.get(page_url, timeout=20)
        response.raise_for_status()
        html_text = response.text

        matches = []
        matches.extend(re.findall(r'<a[^>]+title="[^"]+"[^>]+href="(/may-tinh-xach-tay/[^"]+)"', html_text, flags=re.IGNORECASE))
        matches.extend(re.findall(r'href="(https://fptshop\.com\.vn/may-tinh-xach-tay/[^"]*\?sku=[^"]+|/may-tinh-xach-tay/[^"]*\?sku=[^"]+)"', html_text))
        new_count = 0

        for href in matches:
            full_url = urljoin("https://fptshop.com.vn", href.strip())
            if full_url in seen:
                continue
            seen.add(full_url)
            product_urls.append(full_url)
            new_count += 1

            if max_products is not None and len(product_urls) >= max_products:
                print(f"Đã đạt max_products={max_products}.")
                return product_urls
            if hard_cap is not None and len(product_urls) >= hard_cap:
                print(f"Đã đạt hard_cap={hard_cap} để dừng an toàn.")
                return product_urls

        print(f"Trang {page}: +{new_count} URL sản phẩm, tổng {len(product_urls)}")

        if new_count == 0:
            empty_pages += 1
        else:
            empty_pages = 0

        if empty_pages >= consecutive_empty_pages:
            print(f"Dừng do {consecutive_empty_pages} trang liên tiếp không có URL mới.")
            break

        time.sleep(delay_seconds)

    return product_urls


def extract_product_code(html_text: str):
    normalized = html_text.replace('\\"', '"').replace('\\/', '/')
    # Uu tien ma content id 12 chu so ma frontend dung khi goi comment API.
    match_content_id = re.search(r'"code":"([0-9]{12})","name":"[^"]+","displayName":"[^"]+"', normalized)
    if match_content_id:
        return match_content_id.group(1)

    match = re.search(r'"upc":\{"code":"([^"]+)"\}', normalized)
    if match:
        return match.group(1)

    match_alt = re.search(r'"sku"\s*:\s*"([0-9]{6,})"', normalized)
    if match_alt:
        return match_alt.group(1)

    return None


def _to_int(value):
    if value is None:
        return None
    try:
        return int(float(value))
    except (TypeError, ValueError):
        return None


def _clean_text(value):
    if value is None:
        return None
    cleaned = unescape(str(value))
    cleaned = re.sub(r"<[^>]+>", " ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned or None


def _extract_urls(value):
    if value is None:
        return []
    if isinstance(value, str):
        return [value.strip()] if value.strip() else []
    urls = []
    if isinstance(value, list):
        for item in value:
            if isinstance(item, str) and item.strip():
                urls.append(item.strip())
            elif isinstance(item, dict):
                for key in ("url", "image", "src", "thumb", "thumbnail"):
                    img = item.get(key)
                    if isinstance(img, str) and img.strip():
                        urls.append(img.strip())
                        break
    elif isinstance(value, dict):
        for key in ("url", "image", "src", "thumb", "thumbnail"):
            img = value.get(key)
            if isinstance(img, str) and img.strip():
                urls.append(img.strip())
                break
    return urls


def extract_product_metadata(html_text: str, item_code: str | None = None):
    metadata = {
        "item_id": item_code,
        "product_name": None,
        "brand": None,
        "price": None,
        "final_price": None,
        "rating_count_total": None,
        "image_product": None,
    }

    detail_match = re.search(r'<script[^>]*id="detail-product-script"[^>]*type="application/ld\+json"[^>]*>(.*?)</script>', html_text, flags=re.DOTALL | re.IGNORECASE)
    if detail_match:
        detail_raw = unescape(detail_match.group(1)).strip()
        try:
            detail_candidate = json.loads(detail_raw)
        except json.JSONDecodeError:
            detail_candidate = None
        if isinstance(detail_candidate, dict) and "Product" in ([detail_candidate.get("@type")] if not isinstance(detail_candidate.get("@type"), list) else detail_candidate.get("@type")):
            metadata["product_name"] = detail_candidate.get("name") or metadata["product_name"]
            detail_brand = detail_candidate.get("brand")
            if isinstance(detail_brand, dict):
                metadata["brand"] = detail_brand.get("name") or metadata["brand"]
            elif isinstance(detail_brand, str):
                metadata["brand"] = detail_brand or metadata["brand"]
            detail_offers = detail_candidate.get("offers")
            if isinstance(detail_offers, dict):
                metadata["price"] = metadata["price"] or _to_int(detail_offers.get("price"))
                metadata["final_price"] = metadata["final_price"] or _to_int(detail_offers.get("price"))
            detail_agg = detail_candidate.get("aggregateRating")
            if isinstance(detail_agg, dict):
                metadata["rating_count_total"] = metadata["rating_count_total"] or _to_int(detail_agg.get("reviewCount"))
            detail_image = _extract_urls(detail_candidate.get("image"))
            if detail_image:
                metadata["image_product"] = metadata["image_product"] or detail_image[0]

    ldjson_blocks = re.findall(
        r'<script[^>]*type="application/ld\\+json"[^>]*>(.*?)</script>',
        html_text,
        flags=re.DOTALL | re.IGNORECASE,
    )

    for block in ldjson_blocks:
        raw = unescape(block).strip()
        if not raw:
            continue
        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            continue

        candidates = parsed if isinstance(parsed, list) else [parsed]
        for candidate in candidates:
            if not isinstance(candidate, dict):
                continue
            candidate_type = candidate.get("@type")
            type_list = candidate_type if isinstance(candidate_type, list) else [candidate_type]
            if "Product" not in type_list:
                continue

            metadata["product_name"] = metadata["product_name"] or candidate.get("name")

            brand = candidate.get("brand")
            if isinstance(brand, dict):
                metadata["brand"] = metadata["brand"] or brand.get("name")
            elif isinstance(brand, str):
                metadata["brand"] = metadata["brand"] or brand

            offers = candidate.get("offers")
            if isinstance(offers, dict):
                metadata["price"] = metadata["price"] or _to_int(offers.get("price"))
            elif isinstance(offers, list):
                for offer in offers:
                    if isinstance(offer, dict) and offer.get("price") is not None:
                        metadata["price"] = metadata["price"] or _to_int(offer.get("price"))
                        break

            agg = candidate.get("aggregateRating")
            if isinstance(agg, dict):
                metadata["rating_count_total"] = metadata["rating_count_total"] or _to_int(agg.get("reviewCount"))

            image_urls = _extract_urls(candidate.get("image"))
            if image_urls and not metadata["image_product"]:
                metadata["image_product"] = image_urls[0]

    normalized = html_text.replace('\\"', '"').replace('\\/', '/')

    if not metadata["product_name"]:
        name_match = re.search(r'"upc":\{"code":"[^"]+","name":"([^"]+)"', normalized)
        if name_match:
            metadata["product_name"] = name_match.group(1)

    if not metadata["brand"]:
        brand_match = re.search(r'"brand":\{"code":"[^"]+","name":"([^"]+)"', normalized)
        if brand_match:
            metadata["brand"] = brand_match.group(1)

    if metadata["final_price"] is None:
        final_price_match = re.search(r'"productAdvanceInfo":\{.*?"finalPrice":(\d+)', normalized, flags=re.DOTALL)
        if final_price_match:
            metadata["final_price"] = _to_int(final_price_match.group(1))

    if metadata["price"] is None:
        price_match = re.search(r'"productAdvanceInfo":\{.*?"price":(\d+)', normalized, flags=re.DOTALL)
        if price_match:
            metadata["price"] = _to_int(price_match.group(1))

    if metadata["image_product"] is None:
        img_match = re.search(r'"primaryImage":\{"url":"([^"]+)"\}', normalized)
        if img_match:
            metadata["image_product"] = img_match.group(1)

    if metadata["final_price"] is None:
        metadata["final_price"] = metadata["price"]

    return metadata


def extract_comment_payload(html_text: str):
    normalized = html_text.replace('\\"', '"').replace('\\/', '/')

    pattern = r'"comment":(\{.*?"totalCount":\d+.*?\})\s*,\s*"policyProduct"'
    match = re.search(pattern, normalized, flags=re.DOTALL)
    if not match:
        return {}

    comment_raw = match.group(1)
    try:
        return json.loads(comment_raw)
    except json.JSONDecodeError:
        return {}


def fetch_comment_page(content_id: str, skip_count: int = 0, max_result_count: int = 6, state: list[str] | None = None):
    payload = {
        "content": {"id": str(content_id), "type": "PRODUCT"},
        "state": state or ["ACTIVE"],
        "skipCount": int(skip_count),
        "maxResultCount": int(max_result_count),
        "sortMethod": 1,
    }
    response = SESSION.post(
        COMMENT_API_URL,
        json=payload,
        headers=COMMENT_API_HEADERS,
        timeout=20,
    )
    response.raise_for_status()

    data = response.json()
    if not isinstance(data, dict):
        raise RuntimeError("Phản hồi API comment không đúng định dạng JSON đối tượng.")
    if data.get("status") != 200:
        message = data.get("error", {}).get("message") or data.get("message") or "Lấy comment thất bại."
        raise RuntimeError(message)

    comment_data = data.get("data") or {}
    items = comment_data.get("items") or []
    total_count = _to_int(comment_data.get("totalCount"))
    return items, total_count


def fetch_product_reviews(product_url: str, page_size: int = 6):
    response = SESSION.get(product_url, timeout=20)
    response.raise_for_status()
    html_text = response.text

    item_code = extract_product_code(html_text)
    product_meta = extract_product_metadata(html_text=html_text, item_code=item_code)

    review_items = []
    total_count = None

    if item_code:
        try:
            review_items, total_count = fetch_comment_page(content_id=item_code, skip_count=0, max_result_count=page_size)
        except Exception as exc:
            print(f"Không lấy được comment API cho {product_url}: {exc}")

    if not review_items:
        comment_payload = extract_comment_payload(html_text)
        review_items = comment_payload.get("items") or []
        total_count = total_count or _to_int(comment_payload.get("totalCount")) or len(review_items)

    if product_meta.get("rating_count_total") is None:
        product_meta["rating_count_total"] = _to_int(total_count or len(review_items))

    return product_meta, review_items, total_count, item_code


def parse_rating_row(rating_row: dict, product_meta: dict, product_url: str, row_index: int):
    created_raw = (
        rating_row.get("createdAt")
        or rating_row.get("createAt")
        or rating_row.get("createdDate")
        or rating_row.get("creationTime")
        or rating_row.get("created_time")
        or rating_row.get("time")
    )
    created_at = pd.to_datetime(created_raw, errors="coerce")
    created_at = created_at.isoformat() if pd.notna(created_at) else None

    user_id = (
        rating_row.get("userId")
        or rating_row.get("customerId")
        or rating_row.get("memberId")
        or rating_row.get("fullName")
        or "anonymous"
    )
    cmt_id = rating_row.get("id") or rating_row.get("commentId") or rating_row.get("_id")
    if cmt_id is None:
        fallback = f"{product_url}_{row_index}_{rating_row.get('content', '')}"
        cmt_id = hashlib.md5(fallback.encode("utf-8")).hexdigest()[:16]

    rating_star = rating_row.get("score") or rating_row.get("rating") or rating_row.get("star")
    comment = (
        rating_row.get("content")
        or rating_row.get("comment")
        or rating_row.get("commentContent")
        or ""
    )
    comment = _clean_text(comment) or ""
    review_title = (
        rating_row.get("title")
        or rating_row.get("subject")
        or rating_row.get("headline")
        or rating_row.get("reviewTitle")
        or None
    )

    verified_raw = (
        rating_row.get("verified_purchase")
        or rating_row.get("verifiedPurchase")
        or rating_row.get("isVerified")
        or rating_row.get("isPurchased")
        or rating_row.get("is_buyer")
    )
    verified_purchase = None
    if verified_raw is not None:
        if isinstance(verified_raw, bool):
            verified_purchase = verified_raw
        elif isinstance(verified_raw, (int, float)):
            verified_purchase = bool(verified_raw)
        elif isinstance(verified_raw, str):
            verified_purchase = verified_raw.strip().lower() in {"true", "1", "yes", "y", "co", "có"}

    product_items = (
        rating_row.get("variantName")
        or rating_row.get("modelName")
        or rating_row.get("attributeValue")
        or ""
    )

    reply_content = None
    reply_created_at = None
    reply_user_id = None
    reply_is_admin = None
    reply_like_count = None
    children = rating_row.get("children") or []
    if isinstance(children, list):
        first_reply = next((child for child in children if isinstance(child, dict)), None)
        if first_reply:
            reply_content = _clean_text(first_reply.get("content"))
            reply_raw = first_reply.get("creationTime") or first_reply.get("createdAt") or first_reply.get("createdDate")
            reply_dt = pd.to_datetime(reply_raw, errors="coerce")
            reply_created_at = reply_dt.isoformat() if pd.notna(reply_dt) else None
            reply_user_id = first_reply.get("fullName") or first_reply.get("userId") or first_reply.get("customerId")
            if first_reply.get("isAdministrator") is not None:
                reply_is_admin = bool(first_reply.get("isAdministrator"))
            reply_like_count = first_reply.get("like") or first_reply.get("helpful") or 0

    image_review_urls = []
    for key in ("images", "imageUrls", "image_urls", "attachments", "pictures", "photos", "media"):
        if key in rating_row:
            image_review_urls = _extract_urls(rating_row.get(key))
            if image_review_urls:
                break
    image_review = " | ".join(image_review_urls) if image_review_urls else None

    return {
        "review_id": f"{user_id}_{cmt_id}",
        "shop_id": "FPTShop",
        "item_id": product_meta.get("item_id"),
        "product_url": product_url,
        "product_name": product_meta.get("product_name"),
        "brand": product_meta.get("brand"),
        "price": product_meta.get("price"),
        "final_price": product_meta.get("final_price"),
        "rating_count_total": product_meta.get("rating_count_total"),
        "user_id": str(user_id),
        "rating_star": rating_star,
        "review_title": review_title,
        "comment": comment,
        "verified_purchase": verified_purchase,
        "image_product": product_meta.get("image_product"),
        "image_review": image_review,
        "created_at": created_at,
        "like_count": rating_row.get("like") or rating_row.get("helpful") or 0,
        "product_items": str(product_items).strip(),
        "reply_content": reply_content,
        "reply_created_at": reply_created_at,
        "reply_user_id": reply_user_id,
        "reply_is_admin": reply_is_admin,
        "reply_like_count": reply_like_count,
        "source": "FPTShop"
    }


def crawl_product_reviews(product_url: str, max_reviews: int | None = None, page_size: int = 6, delay_seconds: float = 1.2):
    product_meta, page_reviews, total_count, item_code = fetch_product_reviews(product_url=product_url, page_size=page_size)

    reviews = []
    seen_review_ids = set()
    skip_count = 0
    page_number = 1
    api_entities_seen = 0
    max_retry_empty_page = 1

    while True:
        if max_reviews is not None and len(reviews) >= max_reviews:
            break

        if not page_reviews:
            print(f"Trang {page_number}: không còn comment nào để thu thập.")
            break

        page_added = 0
        page_entities = 0
        for idx, row in enumerate(page_reviews):
            if max_reviews is not None and len(reviews) >= max_reviews:
                break

            children = row.get("children") or []
            child_count = sum(1 for child in children if isinstance(child, dict)) if isinstance(children, list) else 0
            page_entities += 1 + child_count

            parsed = parse_rating_row(row, product_meta=product_meta, product_url=product_url, row_index=skip_count + idx)
            if not parsed["comment"] and not parsed.get("reply_content"):
                continue
            if parsed["review_id"] in seen_review_ids:
                continue

            seen_review_ids.add(parsed["review_id"])
            reviews.append(parsed)
            page_added += 1

        api_entities_seen += page_entities

        total_label = total_count if total_count is not None else "?"
        print(f"Trang {page_number}: +{page_added} comment, tổng {len(reviews)}/{total_label}")

        if total_count is not None and len(reviews) >= total_count:
            break

        if len(page_reviews) < page_size:
            break

        next_skip = skip_count + len(page_reviews)
        page_number += 1

        if max_reviews is not None and len(reviews) >= max_reviews:
            break

        time.sleep(delay_seconds)
        try:
            next_page_reviews, page_total = fetch_comment_page(content_id=item_code, skip_count=next_skip, max_result_count=page_size)
            if total_count is None:
                total_count = page_total

            if not next_page_reviews and total_count is not None and next_skip < total_count:
                for retry_idx in range(max_retry_empty_page):
                    print(f"Trang {page_number}: API trả rỗng bất thường, thử lại lần {retry_idx + 1}/{max_retry_empty_page}.")
                    time.sleep(delay_seconds)
                    retry_page_reviews, _ = fetch_comment_page(content_id=item_code, skip_count=next_skip, max_result_count=page_size)
                    if retry_page_reviews:
                        next_page_reviews = retry_page_reviews
                        break

            skip_count = next_skip
            page_reviews = next_page_reviews
        except Exception as exc:
            print(f"Lỗi khi tải trang comment thứ {page_number}: {exc}")
            break

    print(f"{product_url} -> lấy {len(reviews)} comment hiển thị / totalCount={total_count}")
    if total_count is not None:
        if total_count > api_entities_seen:
            print("Lưu ý: có thể còn dữ liệu chưa lấy do API ẩn bớt bình luận theo trạng thái hiển thị hoặc thay đổi cơ chế phân trang.")
        elif total_count > len(reviews):
            print("Ghi chú: totalCount có thể bao gồm cả phản hồi con (children), nên số comment gốc thu được có thể thấp hơn totalCount.")

    return pd.DataFrame(reviews)

In [10]:
search_url = "https://fptshop.com.vn/tim-kiem?s=laptop&sort=noi-bat&categories=may-tinh-xach-tay"
product_urls = collect_product_urls(
    search_url=search_url,
    max_pages_safety=300,
    max_products=None,
    hard_cap=None,
    consecutive_empty_pages=3,
    delay_seconds=0.8
)

display(product_urls[:10])
print(f"Tổng URL sản phẩm laptop đã thu thập: {len(product_urls)}")

if not product_urls:
    raise ValueError("Không lấy được danh sách product_urls từ trang tìm kiếm FPTShop.")

Trang 1: +24 URL sản phẩm, tổng 24
Trang 2: +24 URL sản phẩm, tổng 48
Trang 3: +24 URL sản phẩm, tổng 72
Trang 4: +24 URL sản phẩm, tổng 96
Trang 5: +24 URL sản phẩm, tổng 120
Trang 6: +24 URL sản phẩm, tổng 144
Trang 7: +24 URL sản phẩm, tổng 168
Trang 8: +24 URL sản phẩm, tổng 192
Trang 9: +24 URL sản phẩm, tổng 216
Trang 10: +24 URL sản phẩm, tổng 240
Trang 11: +24 URL sản phẩm, tổng 264
Trang 12: +24 URL sản phẩm, tổng 288
Trang 13: +24 URL sản phẩm, tổng 312
Trang 14: +24 URL sản phẩm, tổng 336
Trang 15: +24 URL sản phẩm, tổng 360
Trang 16: +24 URL sản phẩm, tổng 384
Trang 17: +24 URL sản phẩm, tổng 408
Trang 18: +24 URL sản phẩm, tổng 432
Trang 19: +24 URL sản phẩm, tổng 456
Trang 20: +24 URL sản phẩm, tổng 480
Trang 21: +24 URL sản phẩm, tổng 504
Trang 22: +24 URL sản phẩm, tổng 528
Trang 23: +24 URL sản phẩm, tổng 552
Trang 24: +24 URL sản phẩm, tổng 576
Trang 25: +24 URL sản phẩm, tổng 600
Trang 26: +24 URL sản phẩm, tổng 624
Trang 27: +24 URL sản phẩm, tổng 648
Trang 28: +24 

['https://fptshop.com.vn/may-tinh-xach-tay/lenovo-thinkpad-x1-carbon-gen-13-u7-258v-21ns010jvn',
 'https://fptshop.com.vn/may-tinh-xach-tay/colorful-rimbook-l1-i5-13420h-a10205500050',
 'https://fptshop.com.vn/may-tinh-xach-tay/macbook-pro-14-m5-pro-2026-15cpu-16gpu-24gb-2tb',
 'https://fptshop.com.vn/may-tinh-xach-tay/macbook-pro-16-m5-max-2026-18cpu-40gpu-64gb-4tb-nano',
 'https://fptshop.com.vn/may-tinh-xach-tay/msi-stealth-16-ai-b3wg-008vn-u9-386h',
 'https://fptshop.com.vn/may-tinh-xach-tay/asus-zenbook-14-ux3405ca-st629w-ultra-7-255h',
 'https://fptshop.com.vn/may-tinh-xach-tay/asus-vivobook-s16-m3607ha-sh186w-ryzen-7-260',
 'https://fptshop.com.vn/may-tinh-xach-tay/lenovo-gaming-loq-15irx10-i7-13645hx-83je01agvn',
 'https://fptshop.com.vn/may-tinh-xach-tay/macbook-air-13-m5-2026-10cpu-8gpu-16gb-512gb',
 'https://fptshop.com.vn/may-tinh-xach-tay/macbook-pro-14-m5-2026-10cpu-10gpu-32gb-2tb-nano-96w']

Tổng URL sản phẩm laptop đã thu thập: 1000


In [11]:
all_frames = []
total_urls = len(product_urls)

for idx, url in enumerate(product_urls, start=1):
    print(f"[{idx}/{total_urls}]")
    df_item = crawl_product_reviews(url, max_reviews=None, page_size=6, delay_seconds=1.2)
    if not df_item.empty:
        all_frames.append(df_item)

if not all_frames:
    raise RuntimeError("Không thu được dữ liệu. Kiểm tra lại URL hoặc thử chạy lại với mạng khác.")

df_raw = pd.concat(all_frames, ignore_index=True)
df_raw = df_raw.drop_duplicates(subset=["review_id"]).reset_index(drop=True)

display(df_raw.head())
display(df_raw.tail())
print(f"Tổng số review thu được: {len(df_raw)}")

[1/1000]
Trang 1: không còn comment nào để thu thập.
https://fptshop.com.vn/may-tinh-xach-tay/lenovo-thinkpad-x1-carbon-gen-13-u7-258v-21ns010jvn -> lấy 0 comment hiển thị / totalCount=0
[2/1000]
Trang 1: +1 comment, tổng 1/1
https://fptshop.com.vn/may-tinh-xach-tay/colorful-rimbook-l1-i5-13420h-a10205500050 -> lấy 1 comment hiển thị / totalCount=1
[3/1000]
Trang 1: không còn comment nào để thu thập.
https://fptshop.com.vn/may-tinh-xach-tay/macbook-pro-14-m5-pro-2026-15cpu-16gpu-24gb-2tb -> lấy 0 comment hiển thị / totalCount=0
[4/1000]
Trang 1: không còn comment nào để thu thập.
https://fptshop.com.vn/may-tinh-xach-tay/macbook-pro-16-m5-max-2026-18cpu-40gpu-64gb-4tb-nano -> lấy 0 comment hiển thị / totalCount=0
[5/1000]
Trang 1: không còn comment nào để thu thập.
https://fptshop.com.vn/may-tinh-xach-tay/msi-stealth-16-ai-b3wg-008vn-u9-386h -> lấy 0 comment hiển thị / totalCount=0
[6/1000]
Trang 1: +1 comment, tổng 1/1
https://fptshop.com.vn/may-tinh-xach-tay/asus-zenbook-14-ux3405ca-s

,review_id,shop_id,item_id,product_url,product_name,brand,price,final_price,rating_count_total,user_id,...,image_review,created_at,like_count,product_items,reply_content,reply_created_at,reply_user_id,reply_is_admin,reply_like_count,source
0,Anh KhoiPD DX_201234,FPTShop,059564510814,https://fptshop.com.vn/may-tinh-xach-tay/color...,Colorful Rimbook L1 i5 13420H (A10205500050),Colorful,14990000,14990000,3,Anh KhoiPD DX,...,None,2026-04-05T07:54:54.114000+00:00,0,,None,None,None,None,None,FPTShop
1,Bình Nguyễn_196901,FPTShop,108185810205,https://fptshop.com.vn/may-tinh-xach-tay/asus-...,Asus Zenbook 14 UX3405CA-ST629W Ultra 7-255H,Asus,34090000,34090000,2,Bình Nguyễn,...,None,2026-03-28T01:12:58.994000+00:00,0,,"Chào anh Bình Dạ, sản phẩm không có màn hình c...",2026-03-28T01:17:57+00:00,Nguyễn Phương Thanh,True,0,FPTShop
2,Bình_210331,FPTShop,320017710671,https://fptshop.com.vn/may-tinh-xach-tay/macbo...,MacBook Air 15 M5 2026 10CPU/10GPU/24GB/1TB,Apple,45990000,45990000,89,Bình,...,None,2026-04-20T14:21:29.756000+00:00,1,,"Chào anh Bình Dạ, cục sạc đi kèm là 35W ạ. Hiệ...",2026-04-20T14:25:41+00:00,Nguyễn Phương Thanh,True,0,FPTShop
3,Thanh_205235,FPTShop,431825326108,https://fptshop.com.vn/may-tinh-xach-tay/macbo...,MacBook Pro 14 M5 Pro 2026 15CPU/16GPU/24GB/1TB,Apple,59990000,59990000,5,Thanh,...,None,2026-04-12T08:04:37.157000+00:00,0,,"Chào anh Thanh, Dạ mẫu này là mẫu M5 Pro thườn...",2026-04-12T08:07:58+00:00,Đông Chí Linh,True,0,FPTShop
4,A Dũng_205578,FPTShop,741367556191,https://fptshop.com.vn/may-tinh-xach-tay/macbo...,MacBook Neo 13 8GB/256GB,Apple,16490000,16490000,24,A Dũng,...,None,2026-04-12T14:04:10.428000+00:00,0,,"Chào anh Dũng, Dạ hiện em kiểm tra thì máy này...",2026-04-12T14:18:43+00:00,Đông Chí Linh,True,0.0,FPTShop


,review_id,shop_id,item_id,product_url,product_name,brand,price,final_price,rating_count_total,user_id,...,image_review,created_at,like_count,product_items,reply_content,reply_created_at,reply_user_id,reply_is_admin,reply_like_count,source
9085,Trung Kiên_43500,FPTShop,760277143745,https://fptshop.com.vn/may-tinh-xach-tay/acer-...,Acer Aspire 3 A315-59-51X8 i5 1235U (NX.K6TSV....,Xiaomi,0,0,6,Trung Kiên,...,None,2024-12-22T02:08:42.290000+00:00,0,,"Chào anh Kiên, Dạ, bên em có hỗ trợ giao hàng ...",2024-12-22T02:14:56+00:00,Nguyễn Thành Đạt,True,0.0,FPTShop
9086,Vũ Hùng Vương_29569,FPTShop,760277143745,https://fptshop.com.vn/may-tinh-xach-tay/acer-...,Acer Aspire 3 A315-59-51X8 i5 1235U (NX.K6TSV....,Xiaomi,0,0,6,Vũ Hùng Vương,...,None,2024-12-13T02:47:46.757000+00:00,0,,NaN,NaN,NaN,None,NaN,FPTShop
9087,Nguyễn Tất Đạt_12577,FPTShop,760277143745,https://fptshop.com.vn/may-tinh-xach-tay/acer-...,Acer Aspire 3 A315-59-51X8 i5 1235U (NX.K6TSV....,Xiaomi,0,0,6,Nguyễn Tất Đạt,...,None,2024-12-03T06:35:31.599000+00:00,0,,Chào anh Đạt Dạ có sẵn hàng nhiều khu vực nhé ...,2024-12-03T06:41:10+00:00,Huỳnh Ngọc Dư,True,0.0,FPTShop
9088,Shushi_4074,FPTShop,760277143745,https://fptshop.com.vn/may-tinh-xach-tay/acer-...,Acer Aspire 3 A315-59-51X8 i5 1235U (NX.K6TSV....,Xiaomi,0,0,6,Shushi,...,None,2024-11-01T00:20:23.767000+00:00,0,,Chào Anh/Chị Shushi Laptop Acer Aspire 3 A315-...,2024-11-01T03:18:38+00:00,Ngô Thị Hồng Ngọc,True,0.0,FPTShop
9089,ân_212163,FPTShop,085911301289,https://fptshop.com.vn/may-tinh-xach-tay/hp-en...,HP ENVY x360 13-bd0528TU i7 1165G7/4Y0Y3PA,Xiaomi,0,0,89,ân,...,None,2026-04-23T15:53:51.158000+00:00,0,,"Chào anh Ân, Dạ sản phẩm này bên em ngừng kinh...",2026-04-24T02:01:01+00:00,Nguyễn Đức Huy,True,0,FPTShop


Tổng số review thu được: 9090


In [14]:
from pathlib import Path

Path("data").mkdir(parents=True, exist_ok=True)
output_path = "data/fptshop_laptop_raw.csv"
df_raw.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Đã lưu dữ liệu thô vào: {output_path}")

Đã lưu dữ liệu thô vào: data/fptshop_laptop_raw.csv
